## Streaming tables — append-only, continuous, declarative

A **streaming table** is an append-only Delta table fed by a Structured Streaming query, declared as a first-class Unity Catalog object instead of imperative `readStream` / `writeStream` code.

```sql
CREATE OR REFRESH STREAMING TABLE fintech_dev.bronze.card_transactions_raw
AS
SELECT * FROM STREAM read_files(
  '/Volumes/fintech_dev/bronze/landing_zone/cards/',
  format => 'json',
  schemaEvolutionMode => 'addNewColumns'
);
```

`read_files(... STREAM ...)` is the **SQL surface of Auto Loader** — same checkpointing and exactly-once semantics you'd get from `cloudFiles`, but the source is just a SQL function call.

**Two pipeline modes drive cost and latency:**

- **Triggered** — the pipeline runs to completion against whatever data is available, then shuts down. The "streaming tables on a schedule" pattern. **Lower cost.**
- **Continuous** — the pipeline runs forever, processing data as it arrives. **Lower latency, higher cost.**

Streaming tables are the **right answer** whenever a question describes *"records arrive continuously and the table should update continuously."* Contrast with a materialized view: an MV recomputes an aggregation on a schedule; a streaming table appends new rows as they land.
